<a href="https://www.kaggle.com/code/kopkritsaikhiao/churn-eda-and-predictions?scriptVersionId=136222055" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Welcome 

Hey everyone, in this project, we'll be focusing on churn prediction. Our goal is to understand and predict churn using different methods. If you find this project interesting, please show your support by upvoting and sharing your feedback.

<div style="text-align: center;">
    <img src="https://img.freepik.com/free-icon/loss_318-699915.jpg?size=626&ext=jpg" alt="Loss Image">
</div>

# Load dataset

In [1]:
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from ydata_profiling import ProfileReport  # pandas_profiling was renamed to ydata_profiling

# On Kaggle the data lives in /kaggle/input; locally, put the CSV in ./data (see README)
DATA_PATH = '/kaggle/input/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print(DATA_PATH)

data/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
import seaborn as sns # For creating plots
import matplotlib.ticker as mtick # For specifying the axes tick format
import matplotlib.pyplot as plt
import plotly.express as px

sns.set(style = 'white')

## preview the head of the dataset

In [3]:
df = pd.read_csv(DATA_PATH)
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Checking all columns names
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [5]:
# Checking the data types of all the columns
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

## use pandas profiling to quickly understand the data

In [6]:
# Pandas profiling before data preprocessing
profile = ProfileReport(df, title='Pandas profiling before data preprcessing', minimal=True, progress_bar=False)
profile.to_notebook_iframe()

## check missing values and drop them

In [7]:
# Converting Total Charges to a numerical data type.
df.TotalCharges = pd.to_numeric(df.TotalCharges, errors='coerce')
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [8]:
# Drop rows with missing values (11 customers with tenure 0 have a blank TotalCharges)
df.dropna(inplace=True)

# Reset the index so it is 0..n-1 again. Without this, later concat(axis=1) calls
# line rows up by the old index labels and mix up different customers.
df.reset_index(drop=True, inplace=True)

In [9]:
# Check columns is dropped
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [10]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7032.000000,7032.000000,7032.000000,7032.000000
mean,0.162400,32.421786,64.798208,2283.300441
std,0.368844,24.545260,30.085974,2266.771362
min,0.000000,1.000000,18.250000,18.800000
25%,0.000000,9.000000,35.587500,401.450000
50%,0.000000,29.000000,70.350000,1397.475000
75%,0.000000,55.000000,89.862500,3794.737500
max,1.000000,72.000000,118.750000,8684.800000


In [11]:
# print all unique value
def summary(df):
    print(f"Dataset has {df.shape[1]} features and {df.shape[0]} examples.")
    print(f"Duplicated rows: {df.duplicated().sum()}")
    summary = pd.DataFrame(index=df.columns)
    summary["Unique"] = df.nunique().values
    summary["Missing"] = df.isnull().sum().values
    summary["Types"] = df.dtypes
    return summary

summary(df)

Dataset has 21 features and 7032 examples.
Duplicated rows: 0


,Unique,Missing,Types
customerID,7032,0,object
gender,2,0,object
SeniorCitizen,2,0,int64
Partner,2,0,object
Dependents,2,0,object
tenure,72,0,int64
PhoneService,2,0,object
MultipleLines,3,0,object
InternetService,3,0,object
OnlineSecurity,3,0,object


In [12]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

# Explore relationships between each features and churn

In [13]:
selected_columns = ['gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod']

for each_name in selected_columns:

    each_count = df.groupby([each_name, 'Churn']).size().reset_index(name='Count')

    fig = px.bar(each_count, x=f'{each_name}', y='Count', color='Churn', barmode='group')

    fig.show()

## what we see from this relationship
Based on the plot, we can observe the following patterns in the churn relationship (about 27% of all customers churn):

* Senior citizens are more likely to churn (42% vs 24%).
* Customers without partners are more likely to churn.
* Customers without dependents are more likely to churn.
* Customers who use fiber optic service are more likely to churn (42%, vs 19% for DSL).
* Customers **without** internet service are the *least* likely to churn (about 7%).
* Customers who have internet but no online security, online backup, device protection, or tech support are more likely to churn.
* Customers who pay month-to-month are much more likely to churn (43%, vs 3% on two-year contracts).
* Customers who use paperless billing and pay with electronic checks are more likely to churn.

Even without machine learning, we can see a clear difference in churn between the groups of each feature.

In [14]:
fig = px.scatter(df, x='MonthlyCharges', y='TotalCharges')

fig.show()

In [15]:
fig = px.box(df, x='Churn', y='tenure')

fig.show()

* this plot we can see that the distribution between who churn and tenure are different 

In [16]:
fig = px.box(df, x='Churn', y='MonthlyCharges')

fig.show()

* this plot we can see that the distribution between who churn and monthly charge are different 

In [17]:
fig = px.box(df, x='Churn', y='TotalCharges')

fig.show()

* The plot shows that customers who churn tend to have lower total charges.

Be careful with this one: TotalCharges is roughly tenure × MonthlyCharges (correlation with tenure is 0.83). Customers who churn are mostly new, so they have not had time to build up a high total. The real signal is mostly tenure, not the total amount billed.

## Based on my initial analysis before using machine learning, here are my simple conclusions and assumptions:

1. New users who do things online, like paying bills and receiving electronic statements, are more likely to stop using the service (churn). This is because they are already comfortable with online processes and may find it easy to switch to a different provider.

2. Customers who don't use online security or protection features tend to choose basic services with lower prices. They prioritize saving money over having additional security measures.

These simple conclusions and assumptions provide insights into potential factors that might affect churn behavior based on customer characteristics and behaviors.

# Machine learning part

In [18]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

# compare two one-hot encodings

In [20]:
df_no_id = df[['gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']]

df_no_id.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [21]:
df_no_id.isna().sum()

gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## pd.get_dummies

In [22]:
# Perform one-hot encoding on selected columns
df_encoded_dummies = pd.get_dummies(df_no_id)

# Display the encoded dataframe
df_encoded_dummies.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_No,Churn_Yes
0,0,1,29.85,29.85,True,False,False,True,True,False,...,False,False,False,True,False,False,True,False,True,False
1,0,34,56.95,1889.50,False,True,True,False,True,False,...,True,False,True,False,False,False,False,True,True,False
2,0,2,53.85,108.15,False,True,True,False,True,False,...,False,False,False,True,False,False,False,True,False,True
3,0,45,42.30,1840.75,False,True,True,False,True,False,...,True,False,True,False,True,False,False,False,True,False
4,0,2,70.70,151.65,True,False,True,False,True,False,...,False,False,False,True,False,False,True,False,False,True


In [23]:
df_object = df[['gender', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'Churn']]

df_object.head()

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Churn
0,Female,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,No
1,Male,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,No
2,Male,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,Yes
3,Male,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),No
4,Female,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,Yes


## OneHotEncoder with sklearn.preprocessing

In [24]:
from sklearn.preprocessing import OneHotEncoder

# Select the columns you want to encode
columns_to_encode = df_object.columns

# Create an instance of the OneHotEncoder class
encoder = OneHotEncoder(sparse_output=False, drop='first')

# Fit and transform the selected columns
encoded_data = encoder.fit_transform(df[columns_to_encode])

# Create a DataFrame with the encoded data (reuse df's index so rows stay aligned)
df_encoded = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(columns_to_encode), index=df.index)

# Concatenate the encoded DataFrame with the original DataFrame
df_final = pd.concat([df.drop(columns=['customerID']+list(columns_to_encode)), df_encoded], axis=1)

# Display the encoded DataFrame
df_final.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,1,29.85,29.85,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
1,0,34,56.95,1889.50,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0,2,53.85,108.15,1.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
3,0,45,42.30,1840.75,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0,2,70.70,151.65,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0


In [25]:
len(df_encoded_dummies.columns), len(df_final.columns)

(47, 31)

* `pd.get_dummies` makes one column per category, so every feature gets one redundant column (for example `gender_Female` and `gender_Male` always add up to 1).

* `OneHotEncoder(drop='first')` drops one column per feature, which removes that redundancy and gives fewer dimensions (31 vs 47 columns).

In [26]:
# Compare column sets
columns_diff = set(df_encoded_dummies.columns) - set(df_final.columns)

# Print the differing columns
print("Columns that are present in df_encoded_dummies but not in df_final:")
for column in sorted(columns_diff):
    print(column)

# Repeat the comparison for columns present in df_final but not in df_encoded_dummies
columns_diff = set(df_final.columns) - set(df_encoded_dummies.columns)
print("Columns that are present in df_final but not in df_encoded_dummies:")
for column in sorted(columns_diff):
    print(column)

Columns that are present in df_encoded_dummies but not in df_final:
Churn_No
Contract_Month-to-month
Dependents_No
DeviceProtection_No
InternetService_DSL
MultipleLines_No
OnlineBackup_No
OnlineSecurity_No
PaperlessBilling_No
Partner_No
PaymentMethod_Bank transfer (automatic)
PhoneService_No
StreamingMovies_No
StreamingTV_No
TechSupport_No
gender_Female
Columns that are present in df_final but not in df_encoded_dummies:


The Shapiro-Wilk test checks whether a column looks normally distributed. The statistic W is between 0 and 1; values close to 1 mean a good fit to a normal distribution.

Only the continuous columns are worth testing. The one-hot columns are 0/1, so they can never be normal. This result helps pick a scaler: when features are not normal and have outliers, MinMax and Standard scaling behave differently.

In [27]:
from scipy.stats import shapiro

# Check for Gaussian distribution in the continuous columns
for column in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    shapiro_stat, p_value = shapiro(df[column])
    print(f"{column:15s} W={shapiro_stat:.3f}  p={p_value:.3g}  Gaussian: {p_value > 0.05}")

tenure          W=0.904  p=8.32e-55  Gaussian: False
MonthlyCharges  W=0.921  p=2.43e-51  Gaussian: False
TotalCharges    W=0.860  p=1.05e-61  Gaussian: False


In [28]:
df_encoded_dummies.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes',
       'Dependents_No', 'Dependents_Yes', 'PhoneService_No',
       'PhoneService_Yes', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year',
       'Contract_Two year', '

## train test split

In [29]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Define the features and target variable
features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Female', 'gender_Male', 'Partner_No', 'Partner_Yes',
       'Dependents_No', 'Dependents_Yes', 'PhoneService_No',
       'PhoneService_Yes', 'MultipleLines_No',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No', 'OnlineBackup_No internet service',
       'OnlineBackup_Yes', 'DeviceProtection_No',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No', 'StreamingMovies_No internet service',
       'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year',
       'Contract_Two year', 'PaperlessBilling_No', 'PaperlessBilling_Yes',
       'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

target = ['Churn_Yes']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(df_encoded_dummies[features], np.array(df_encoded_dummies[target]).ravel(),
                                                    test_size=0.2, random_state=42, stratify=df_encoded_dummies[target])
print("total features :", len(df_encoded_dummies[features].columns))
print("X \n", X_train.iloc[0], "\n y \n", y_train[0])

total features : 45
X 
 SeniorCitizen                                    0
tenure                                          65
MonthlyCharges                               94.55
TotalCharges                               6078.75
gender_Female                                False
gender_Male                                   True
Partner_No                                   False
Partner_Yes                                   True
Dependents_No                                False
Dependents_Yes                                True
PhoneService_No                              False
PhoneService_Yes                              True
MultipleLines_No                             False
MultipleLines_No phone service               False
MultipleLines_Yes                             True
InternetService_DSL                          False
InternetService_Fiber optic                   True
InternetService_No                           False
OnlineSecurity_No                            False
OnlineS

In [30]:
y_train

array([False, False, False, ..., False, False,  True])

In [31]:
y_train = y_train.ravel()
y_test = y_test.ravel()

y_train

array([False, False, False, ..., False, False,  True])


## evaluation metrics and a baseline

About 73% of customers do not churn, so a model that always says "No churn" already gets about 73% accuracy. Accuracy alone can hide a model that never finds churners, so every model below reports:

* **Accuracy**: share of all predictions that are correct
* **ROC-AUC**: how well the model ranks churners above non-churners (0.5 = random, 1.0 = perfect)
* **Recall**: share of real churners the model catches
* **F1**: balance between recall and precision for the churn class


In [32]:

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score

results = {}

def evaluate(model, X, y):
    y_pred = model.predict(X)
    # ROC-AUC needs a score, not a hard 0/1 label
    if hasattr(model, 'predict_proba'):
        y_score = model.predict_proba(X)[:, 1]
    else:
        y_score = model.decision_function(X)
    return {
        'Accuracy': accuracy_score(y, y_pred),
        'ROC-AUC': roc_auc_score(y, y_score),
        'Recall (churn)': recall_score(y, y_pred),
        'F1 (churn)': f1_score(y, y_pred),
    }

baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
results['Baseline (always "No")'] = evaluate(baseline, X_test, y_test)
results['Baseline (always "No")']


{'Accuracy': 0.7341862117981521,
 'ROC-AUC': 0.5,
 'Recall (churn)': 0.0,
 'F1 (churn)': 0.0}

# Logistic Regression

## Logistic Regression with pd.get_dummies

In [33]:
# Create the pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),      # Apply feature scaling using MinMaxScaler
    ('classifier', LogisticRegression(max_iter=1000))    # Logistic Regression classifier
])

# Fit the pipeline on the training data

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', MinMaxScaler()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [34]:
# Predict on the test data and evaluate the model
results['LogReg (get_dummies, MinMax)'] = evaluate(pipeline, X_test, y_test)
results['LogReg (get_dummies, MinMax)']

{'Accuracy': 0.8024164889836531,
 'ROC-AUC': 0.8348134554358574,
 'Recall (churn)': 0.56951871657754,
 'F1 (churn)': 0.6051136363636364}

In [35]:
df_final.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check',
       'Churn_Yes'],
      dtype='object')

## Logistic Regression with sklearn one-hot-encoder

In [36]:
# Should be all zeros now that df's index was reset before encoding
df_final.isna().sum().sum()

0

In [37]:
# Define the features and target variable
features = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']

target = ['Churn_Yes']

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(df_final[features], df_final[target],
                                                    test_size=0.2, random_state=42, stratify=df_final[target])
print("total features :", len(df_encoded_dummies[features].columns))
print("X \n", X_train.iloc[0], "\n y \n", y_train)

total features : 30
X 
 SeniorCitizen                               0.00
tenure                                     65.00
MonthlyCharges                             94.55
TotalCharges                             6078.75
gender_Male                                 1.00
Partner_Yes                                 1.00
Dependents_Yes                              1.00
PhoneService_Yes                            1.00
MultipleLines_No phone service              0.00
MultipleLines_Yes                           1.00
InternetService_Fiber optic                 1.00
InternetService_No                          0.00
OnlineSecurity_No internet service          0.00
OnlineSecurity_Yes                          1.00
OnlineBackup_No internet service            0.00
OnlineBackup_Yes                            1.00
DeviceProtection_No internet service        0.00
DeviceProtection_Yes                        1.00
TechSupport_No internet service             0.00
TechSupport_Yes                             1

In [38]:
y_train = np.array(y_train).ravel()
y_test = np.array(y_test).ravel()

y_train

array([0., 0., 0., ..., 0., 0., 1.])

In [39]:
# Create the pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),      # Apply feature scaling using MinMaxScaler
    ('classifier', LogisticRegression(max_iter=1000))    # Logistic Regression classifier
])

# Fit the pipeline on the training data

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', MinMaxScaler()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [40]:
# Predict on the test data and evaluate the model
results['LogReg (OneHotEncoder, MinMax)'] = evaluate(pipeline, X_test, y_test)
results['LogReg (OneHotEncoder, MinMax)']

{'Accuracy': 0.8031272210376688,
 'ROC-AUC': 0.8349351093072976,
 'Recall (churn)': 0.56951871657754,
 'F1 (churn)': 0.6059743954480796}

### results

* `pd.get_dummies` (45 features): ROC-AUC 0.835, accuracy 0.802
* sklearn `OneHotEncoder(drop='first')` (30 features): ROC-AUC 0.835, accuracy 0.803

Both encodings give practically the same result. The extra columns from `get_dummies` are redundant, but logistic regression in scikit-learn is regularized by default, so they don't hurt.

(An earlier version of this notebook showed a bigger gap, 0.787 vs 0.801. That gap came from a bug: the rows got misaligned when the encoded columns were joined back on, plus the model not converging. Both are fixed above.)

This is the formula for logistic regression:

p(x) = 1 / (1 + e^(-z))

With many features, z looks like a multiple linear regression:

z = β₀ + β₁x₁ + β₂x₂ + ... + βₚxₚ

Fewer dimensions still makes the model simpler to read, so we keep the `OneHotEncoder` version for the rest of the notebook.

In [41]:
from sklearn.preprocessing import StandardScaler

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Apply feature scaling using StandardScaler
    ('classifier', LogisticRegression(max_iter=1000))    # Logistic Regression classifier
])

# Fit the pipeline on the training data

pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', LogisticRegression(max_iter=1000))])

In [42]:
# Predict on the test data and evaluate the model
results['LogReg (OneHotEncoder, Standard)'] = evaluate(pipeline, X_test, y_test)
results['LogReg (OneHotEncoder, Standard)']

{'Accuracy': 0.8038379530916845,
 'ROC-AUC': 0.8356727976766699,
 'Recall (churn)': 0.5748663101604278,
 'F1 (churn)': 0.6090651558073654}

## change from minmax scale to standard scale

* Min-max scaling, also known as normalization, is a data scaling technique that transforms the features to a specific range, typically between 0 and 1. It achieves this by subtracting the minimum value from each feature and then dividing it by the range (maximum value minus minimum value).

* standard scaling, also known as z-score normalization, transforms the features to have a mean of 0 and a standard deviation of 1. It achieves this by subtracting the mean from each feature and then dividing it by the standard deviation.

In simpler terms, min-max scaling scales the features to a specific range, while standard scaling standardizes the features to have a mean of 0 and a standard deviation of 1.

For logistic regression the choice of scaler makes almost no difference here (see the table above).

In [43]:
import plotly.express as px

# Retrieve the logistic regression classifier from the pipeline
logreg = pipeline.named_steps['classifier']

# Get the coefficients of the logistic regression model
coef = logreg.coef_[0]

# Create a dataframe of variable names and corresponding coefficients
weights_df = pd.DataFrame({'Variable': features, 'Weight': coef})

# Sort the weights in descending order and select the top 10
top_weights = weights_df.sort_values('Weight', ascending=False).head(10)

# Create the bar chart using Plotly
fig = px.bar(top_weights, x='Variable', y='Weight', title='Top 10 Positive Variable Weights')
fig.show()

In [44]:
import plotly.express as px

# Retrieve the logistic regression classifier from the pipeline
logreg = pipeline.named_steps['classifier']

# Get the coefficients of the logistic regression model
coef = logreg.coef_[0]

# Create a dataframe of variable names and corresponding coefficients
weights_df = pd.DataFrame({'Variable': features, 'Weight': coef})

# Sort the weights in ascending order and select the top 10
top_weights = weights_df.sort_values('Weight', ascending=True).head(10)

# Create the bar chart using Plotly
fig = px.bar(top_weights, x='Variable', y='Weight', title='Top 10 Negative Variable Weights')
fig.show()

Positive weights push a customer towards churning and negative weights push them towards staying. Because the features are standardized, the size of a weight shows how strongly that feature matters.

* **Towards staying:** long tenure (the strongest effect of all), one- or two-year contracts, online security and tech support.
* **Towards churning:** fiber optic internet, TotalCharges, streaming services, and paying by electronic check.

MonthlyCharges shows up with a *negative* weight even though churners pay more per month. This happens because MonthlyCharges is mostly explained by other columns (fiber, streaming, multiple lines), so the model splits the effect between them. With correlated features like these, look at the group of related features together rather than reading one weight on its own.

Overall this matches what we saw in the EDA.

# Random Forest

Random Forest is a machine learning algorithm that combines multiple decision trees to make predictions. It is called "random" because it creates each decision tree using a random subset of the training data and a random subset of the features.

Think of it like a "forest" of decision trees, where each tree independently predicts the outcome, and the final prediction is made by averaging or voting across all the individual tree predictions.

Random Forest is known for its simplicity and ease of understanding. It can handle both classification and regression tasks and is effective in handling large datasets with many features. It is also robust to outliers and can provide insights into feature importance.

In simple terms, Random Forest is like a team of decision trees working together to make accurate predictions. Its randomness and combination of trees make it a powerful and easy-to-understand algorithm for machine learning tasks.

<div style="text-align: center;">
    <img src="https://img.freepik.com/free-photo/aerial-view-vibrant-green-trees-forest_181624-49828.jpg?size=626&ext=jpg" alt="forrest Image">
</div>

In [45]:
from sklearn.ensemble import RandomForestClassifier

# Create the pipeline with Random Forest Classifier (tree models do not need scaling)
pipeline = Pipeline([
    ('classifier', RandomForestClassifier(random_state=42))    # Random Forest Classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

Pipeline(steps=[('classifier', RandomForestClassifier(random_state=42))])

In [46]:
# Predict on the test data and evaluate the model
results['Random Forest (default)'] = evaluate(pipeline, X_test, y_test)
results['Random Forest (default)']

{'Accuracy': 0.7896233120113717,
 'ROC-AUC': 0.8164903116927489,
 'Recall (churn)': 0.5187165775401069,
 'F1 (churn)': 0.5672514619883041}

In [47]:
# Retrieve the feature importances from the trained Random Forest model
importances = pipeline.named_steps['classifier'].feature_importances_

# Create a dataframe of variable names and corresponding feature importances
importance_df = pd.DataFrame({'Variable': features, 'Importance': importances})

# Sort the feature importances in descending order
importance_df = importance_df.sort_values('Importance', ascending=False)

# Create the bar chart using Plotly
fig = px.bar(importance_df, x='Variable', y='Importance', title='Feature Importance')
fig.show()

In [48]:
from sklearn.model_selection import cross_val_score

# Define the pipeline
pipeline = Pipeline([
    ('classifier', RandomForestClassifier(n_estimators=1000,
                                          oob_score=True,
                                          n_jobs=-1,
                                          random_state=50,
                                          max_features='sqrt',
                                          max_leaf_nodes=30))  # Random Forest classifier
])

# Perform cross-validation
scores = cross_val_score(estimator=pipeline, X=X_train, y=y_train, cv=5, scoring='roc_auc')

# Print scores and their mean
print("ROC-AUC each fold :", scores)
print("Mean:", scores.mean())

ROC-AUC each fold : [0.83950335 0.86449586 0.84676525 0.82357657 0.84680169]
Mean: 0.8442285422757051


In [49]:
# Defining a parameter grid for hyperparameter tuning with different values to be tested for 'n_estimators', 'max_depth', and 'max_features' hyperparameters
param_grid = [{'n_estimators': [100, 200, 300], 'max_depth': [None,2,3,10,20], 'max_features': ['sqrt',2,4,8,16,'log2', None]}]

### tuning hyper-parameter

In [50]:
from sklearn.model_selection import GridSearchCV

# Creating a random forest classifier object 'temp_rf' with a random state of 0 and parallel processing enabled
temp_rf=RandomForestClassifier(random_state=0,n_jobs=-1)
# Creating a grid search object 'grid_search' using the 'GridSearchCV' function, with a random forest classifier as the estimator, a parameter grid, 'roc_auc' as the scoring metric, and 5-fold cross-validation with parallel processing
grid_search=GridSearchCV(estimator=temp_rf, param_grid=param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
# Performing grid search on the training data to find the best hyperparameters for the model
grid_search.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(n_jobs=-1, random_state=0),
             n_jobs=-1,
             param_grid=[{'max_depth': [None, 2, 3, 10, 20],
                          'max_features': ['sqrt', 2, 4, 8, 16, 'log2', None],
                          'n_estimators': [100, 200, 300]}],
             scoring='roc_auc')

In [51]:
# Best cross-validated ROC-AUC found by Grid Search
grid_search.best_score_

0.8423076923076923

In [52]:
# Retrieving the best parameter values found by the grid search
grid_search.best_params_

{'max_depth': 10, 'max_features': 'sqrt', 'n_estimators': 300}

In [53]:

# Evaluate the best grid-search model on the held-out test set
pipeline = grid_search.best_estimator_
results['Random Forest (grid search)'] = evaluate(pipeline, X_test, y_test)
results['Random Forest (grid search)']


{'Accuracy': 0.7960199004975125,
 'ROC-AUC': 0.8335244420746385,
 'Recall (churn)': 0.5080213903743316,
 'F1 (churn)': 0.5697151424287856}

# Support Vector Machine (SVC)

In [54]:
from sklearn.svm import SVC

# Create the pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),      # Apply feature scaling using StandardScaler
    ('classifier', SVC())              # Support Vector Machine classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('classifier', SVC())])

In [55]:
# Predict on the test data and evaluate the model
results['SVC'] = evaluate(pipeline, X_test, y_test)
results['SVC']

{'Accuracy': 0.7867803837953091,
 'ROC-AUC': 0.7908575821422471,
 'Recall (churn)': 0.4919786096256685,
 'F1 (churn)': 0.5508982035928144}

# AdaBoost


AdaBoost, short for Adaptive Boosting, is a machine learning algorithm that combines multiple weak or simple models (often decision trees) to create a strong predictive model. It is an ensemble method that aims to improve the performance of individual models by sequentially training them on different weighted versions of the dataset.

The key idea behind AdaBoost is to give more importance to the misclassified instances in each iteration, allowing subsequent models to focus on the difficult cases. In each iteration, the algorithm assigns higher weights to misclassified instances and lower weights to correctly classified instances. This way, subsequent models will pay more attention to the challenging observations.

During the prediction phase, AdaBoost combines the predictions from all the weak models through a weighted voting or weighted averaging approach to make the final prediction.

AdaBoost is known for its ability to handle complex datasets and achieve high accuracy. It is especially effective when combined with weak models, as it can effectively learn from their collective knowledge and generalize well to new, unseen data.

In simple terms, AdaBoost is a technique that combines multiple weak models to create a strong and accurate model. It learns from mistakes and gives more emphasis to challenging instances, resulting in improved performance.


<div style="text-align: center;">
    <img src="https://static.packt-cdn.com/products/9781788295758/graphics/image_04_046-1.png" alt="adaboost Image">
</div>

In [56]:
from sklearn.ensemble import AdaBoostClassifier

# Create the pipeline (tree-based, so no scaling needed)
pipeline = Pipeline([
    ('classifier', AdaBoostClassifier(algorithm='SAMME', random_state=42))       # AdaBoost Classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

Pipeline(steps=[('classifier',
                 AdaBoostClassifier(algorithm='SAMME', random_state=42))])

In [57]:
# Predict on the test data and evaluate the model
results['AdaBoost'] = evaluate(pipeline, X_test, y_test)
results['AdaBoost']

{'Accuracy': 0.7953091684434968,
 'ROC-AUC': 0.8408301453116669,
 'Recall (churn)': 0.5026737967914439,
 'F1 (churn)': 0.5662650602409639}

# XGBoost

XGBoost, short for eXtreme Gradient Boosting, is a powerful machine learning algorithm known for its high performance and efficiency. It belongs to the family of gradient boosting algorithms and is designed to optimize model performance through an iterative boosting process.

XGBoost utilizes an ensemble of decision trees to make predictions. It builds each tree in a sequential manner, with each subsequent tree focusing on correcting the mistakes of the previous trees. The algorithm assigns higher weights to the misclassified instances, allowing subsequent trees to pay more attention to those instances.

What sets XGBoost apart is its advanced regularization techniques, which help prevent overfitting and improve generalization. It incorporates both L1 and L2 regularization to control the complexity of the trees and includes a term to penalize complex models.

XGBoost also features a unique approximation algorithm that speeds up the training process by considering only the most informative splits. It uses parallel processing capabilities and various optimization strategies to efficiently handle large datasets.

The algorithm offers flexibility with customizable parameters that allow fine-tuning for optimal performance. XGBoost is widely used across various domains and has achieved remarkable success in machine learning competitions.

In simple terms, XGBoost is an advanced gradient boosting algorithm that combines decision trees in an iterative process to make accurate predictions. It incorporates regularization techniques and optimization strategies to improve performance, making it a popular choice in machine learning tasks.

<div style="text-align: center;">
    <img src="https://www.researchgate.net/publication/345327934/figure/fig3/AS:1022810793209856@1620868504478/Flow-chart-of-XGBoost.png" alt="XGBoost Image">
</div>

In [58]:
from xgboost import XGBClassifier

# Create the pipeline (tree-based, so no scaling needed)
pipeline = Pipeline([
    ('classifier', XGBClassifier(random_state=42))            # XGBoost Classifier
])

# Fit the pipeline on the training data
pipeline.fit(X_train, y_train)

Pipeline(steps=[('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               early_stopping_rounds=None,
                               enable_categorical=False, eval_metric=None,
                               feature_types=None, gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, random_state=42, ...))])

In [59]:
# Predict on the test data and evaluate the model
results['XGBoost (default)'] = evaluate(pipeline, X_test, y_test)
results['XGBoost (default)']

{'Accuracy': 0.7668798862828714,
 'ROC-AUC': 0.8141801823255044,
 'Recall (churn)': 0.516042780748663,
 'F1 (churn)': 0.5406162464985994}


## tuned XGBoost

The default XGBoost grows deep trees with no limit on the number of rounds, so it overfits this small dataset. A few conservative settings fix that.


In [60]:

pipeline = Pipeline([
    ('classifier', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                                 subsample=0.8, colsample_bytree=0.8, random_state=42))
])
pipeline.fit(X_train, y_train)

results['XGBoost (tuned)'] = evaluate(pipeline, X_test, y_test)
results['XGBoost (tuned)']


{'Accuracy': 0.7889125799573561,
 'ROC-AUC': 0.8359614020738103,
 'Recall (churn)': 0.5240641711229946,
 'F1 (churn)': 0.5689404934687954}

# Model comparison

In [61]:

comparison = pd.DataFrame(results).T.round(3)
comparison.sort_values('ROC-AUC', ascending=False)


,Accuracy,ROC-AUC,Recall (churn),F1 (churn)
AdaBoost,0.795,0.841,0.503,0.566
"LogReg (OneHotEncoder, Standard)",0.804,0.836,0.575,0.609
XGBoost (tuned),0.789,0.836,0.524,0.569
"LogReg (get_dummies, MinMax)",0.802,0.835,0.570,0.605
"LogReg (OneHotEncoder, MinMax)",0.803,0.835,0.570,0.606
Random Forest (grid search),0.796,0.834,0.508,0.570
Random Forest (default),0.790,0.816,0.519,0.567
XGBoost (default),0.767,0.814,0.516,0.541
SVC,0.787,0.791,0.492,0.551
"Baseline (always ""No"")",0.734,0.500,0.000,0.000


### what the comparison shows

* Every model clearly beats the baseline. The baseline gets 73% accuracy but catches **zero** churners, which is why accuracy alone is misleading here.
* The best models are very close: AdaBoost, logistic regression, tuned XGBoost and the tuned random forest all score ROC-AUC 0.83 to 0.84. With test-set differences this small, **logistic regression** is a good choice: it is simple, fast, and its weights are easy to explain.
* Default XGBoost and the default random forest overfit; limiting tree depth (tuning) fixed most of that.
* All models catch only about half of the churners (recall 0.50 to 0.57). If catching churners matters more than false alarms, lower the decision threshold below 0.5 or train with `class_weight='balanced'`.

# Differences between each model

Random Forest, AdaBoost, and XGBoost are all ensemble learning methods that combine multiple weak models to create a strong predictive model.

However, there are some key differences between them:

1. Random Forest:

* It builds multiple decision trees independently and combines their predictions through voting or averaging.
* Each tree is trained on a random subset of the training data and a random subset of features.
* Random Forest is known for its simplicity, robustness against overfitting, and ability to handle high-dimensional data.
* It performs well in a wide range of tasks and is relatively easy to interpret.

2. AdaBoost (Adaptive Boosting):

* It trains weak models sequentially, with each subsequent model focusing on correcting the mistakes of the previous models.
* Instances that are misclassified by previous models are given higher weights, allowing subsequent models to pay more attention to them.
* AdaBoost adapts to difficult cases by boosting the importance of misclassified instances.
* It is effective in handling complex datasets and achieving high accuracy.

3. XGBoost (eXtreme Gradient Boosting):

* It is an optimized implementation of gradient boosting, designed for speed and performance.
* XGBoost uses a combination of gradient descent optimization and regularization techniques.
* It includes advanced regularization techniques, parallel processing capabilities, and optimization strategies.
* XGBoost achieves high accuracy, handles large datasets efficiently, and is widely used in machine learning competitions.


In summary, Random Forest focuses on combining independently trained decision trees, AdaBoost adapts to challenging cases by boosting the importance of misclassified instances, and XGBoost is an optimized implementation of gradient boosting with advanced regularization and optimization techniques.

# Thank you for attention

Thank you for taking the time to read and engage with this information! If you found it helpful or enjoyable, please consider giving it an upvote. Positive feedback like this greatly motivates me to continue exploring new areas and sharing my knowledge.